# Project 06: Hybrid Retrieval and Evaluation for RAG

Improve a basic RAG search by combining keyword search (BM25) with vector search, then measure how often the right text is found.

Run top to bottom. Each step matches the README.

In [ ]:
import os, subprocess
REPO = "TharunPatel20/GenAI-Engineer"   # replace with your GitHub repo, e.g. "your-name/ai-engineer-course"
PROJECT = "projects/06-rag-advanced"
if REPO.startswith("<"):
    print("Repo not set. Upload the project files with the Files panel (left sidebar), into a folder named 06-rag-advanced, then run os.chdir('06-rag-advanced') before Step 1.")
else:
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"], check=True)
    os.chdir(REPO.split("/")[1] + "/" + PROJECT)

## Step 1: install the pinned packages

This cell installs the exact versions from `requirements.txt`. Colab may ask you to restart the session. If it does, click **Restart session**, then run the cells again from the top (Step 0 onward).

In [ ]:
%pip install -q langchain-core==0.3.72 langchain-community==0.3.27 langchain-google-genai==2.1.8 langchain-text-splitters==0.3.9 faiss-cpu==1.11.0 rank-bm25==0.2.2 python-dotenv==1.1.1

## Step 2: API key (Colab Secrets)

1. Click the key icon in the left sidebar (**Secrets**).
2. Click **Add new secret**. Name it exactly `GEMINI_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for that secret.

Never paste the key into a cell. The BM25 keyword part works without a key. The vector part and the evaluation need one.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. Add GEMINI_API_KEY in Colab Secrets, then run this cell again. Details:", e)
print("key found:", bool(os.environ.get("GEMINI_API_KEY")))

### Step 1: Split the FAQ into chunks
**What:** `load_chunks()` reads the FAQ and splits it into chunks of up to 300 characters.
**Why:** Retrieval returns chunks, not whole files. Each FAQ entry is short, so a chunk usually holds one or two entries.

In [ ]:
import os
import re
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

PROJECT_DIR = Path.cwd()  # notebook version: the project folder is the current folder (the script uses Path(__file__).parent)
DOCS_DIR = PROJECT_DIR / "sample_docs"
# Model names change, so check Google AI Studio if a name stops working.
CHAT_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "models/gemini-embedding-001")


def tokenize(text):
    # Keeps codes such as "acr-4471" as one token, so an exact code can match exactly.
    return re.findall(r"[a-z0-9][a-z0-9\-]*", text.lower())


def load_chunks(chunk_size=300, chunk_overlap=0):
    text = (DOCS_DIR / "faq.txt").read_text(encoding="utf-8")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.create_documents([text], metadatas=[{"source": "faq.txt"}])
    for i, doc in enumerate(chunks):
        doc.metadata["chunk_id"] = i
    return chunks


chunks = load_chunks()
print(len(chunks), "chunks")
print(chunks[0].page_content)

### Step 2: Keyword search with BM25
**What:** `keyword_ranking()` scores every chunk with BM25 and returns the chunk ids that share at least one word with the question.
**Why:** BM25 rewards rare, exact words. A code like `ACR-4471` appears in only one chunk, so BM25 finds it. Vector search can miss it, because embeddings capture meaning more than exact strings.

The tokenizer keeps codes such as `acr-4471` as one token. Without that, the code would split into pieces and match too loosely.

In [ ]:
from rank_bm25 import BM25Okapi


def get_api_key():
    key = os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        raise SystemExit(
            "GEMINI_API_KEY is missing. Copy .env.example to .env and paste your key after GEMINI_API_KEY="
        )
    return key


class HybridRetriever:
    def __init__(self):
        self.chunks = load_chunks()
        self.bm25 = BM25Okapi([tokenize(c.page_content) for c in self.chunks])
        self._vector_store = None  # built on the first vector search, so keyword search needs no key

    def _vectors(self):
        if self._vector_store is None:
            embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=get_api_key())
            self._vector_store = FAISS.from_documents(self.chunks, embeddings)
        return self._vector_store

    def keyword_ranking(self, query):
        # Returns chunk ids, best first. Chunks with no matching word are left out.
        scores = self.bm25.get_scores(tokenize(query))
        ranked = sorted(range(len(self.chunks)), key=lambda i: scores[i], reverse=True)
        return [i for i in ranked if scores[i] > 0]

    def vector_ranking(self, query, pool=10):
        # FAISS returns the closest chunks first. We only use the order, not the raw distance.
        pool = min(pool, len(self.chunks))
        docs = self._vectors().similarity_search(query, k=pool)
        return [d.metadata["chunk_id"] for d in docs]

    def search(self, query, k=3, mode="hybrid", pool=10):
        if mode == "keyword":
            ids = self.keyword_ranking(query)
        elif mode == "vector":
            ids = self.vector_ranking(query, pool)
        elif mode == "hybrid":
            ids = reciprocal_rank_fusion([self.keyword_ranking(query), self.vector_ranking(query, pool)])
        else:
            raise ValueError(f"Unknown mode: {mode}. Use keyword, vector, or hybrid.")
        return [self.chunks[i] for i in ids[:k]]


retriever = HybridRetriever()
for doc in retriever.search("What does order code ACR-4471 include?", mode="keyword"):
    print(doc.page_content[:80])

### Step 3: Vector search with FAISS and Gemini embeddings
**What:** The first vector search embeds all chunks with `gemini-embedding-001`, stores them in FAISS, and returns the closest chunks.
**Why:** Vectors match on meaning. A question such as "my lamp is dead" can find "try a different outlet" even though the words differ.

This step needs a key. The code uses only the order of the results. FAISS returns a distance, and the direction of that distance depends on the index type. Read the [LangChain FAISS documentation](https://python.langchain.com/docs/integrations/vectorstores/faiss/) before you use raw scores.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

for doc in retriever.search("My lamp is dead and nothing happens", mode="vector"):
    print(doc.page_content[:80])

### Step 4: Hybrid search with reciprocal rank fusion
**What:** `reciprocal_rank_fusion()` takes the BM25 list and the vector list. Each chunk gets `1 / (60 + rank)` from each list, and the scores are added.
**Why:** RRF does not need the two scores to be on the same scale. A chunk that ranks high in both lists rises to the top. A chunk that only one method finds still appears.

Hybrid retrieval is the combination from the course recap: keyword search for exact terms, and vector search for meaning. See [recap-2026-11-11.md](../../claude-files/recap-2026-11-11.md). This step needs a key.

In [ ]:
RRF_CONSTANT = 60  # the usual constant in reciprocal rank fusion


def reciprocal_rank_fusion(rankings, k=RRF_CONSTANT):
    # Each list gives an item 1 / (k + rank). Items found by several lists add up.
    scores = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


for doc in retriever.search("Can I charge it from a battery pack?", mode="hybrid"):
    print(doc.page_content[:80])

### Step 6: Reranking (placeholder)
**What:** `rerank_by_overlap()` re-sorts the results by how many query words each chunk contains.
**Why:** The first search returns candidates. A reranker puts the best candidate first. This placeholder is simple and only shows where a reranker fits. A real reranker is a **cross-encoder**, a model that reads the question and each chunk together and gives one relevance score. Building one is a stretch exercise.

In [ ]:
# PLACEHOLDER reranker. It scores each chunk by how many query words it contains.
# A real cross-encoder reranker (a model that reads the query and chunk together) is a stretch exercise.
def rerank_by_overlap(query, docs):
    query_terms = set(tokenize(query))

    def overlap(doc):
        return len(query_terms & set(tokenize(doc.page_content)))

    return sorted(docs, key=overlap, reverse=True)


candidates = retriever.search("Is there a student discount?", k=5, mode="hybrid")
for doc in rerank_by_overlap("Is there a student discount?", candidates)[:3]:
    print(doc.page_content[:80])

### Step 7: Measure retrieval hit rate
**What:** Load `eval_questions.json`. Each question has an `expected_phrase`. For each mode, it checks whether any of the top 3 chunks contains that phrase.
**Why:** You cannot improve retrieval without a way to measure it. Hit rate answers "did the right text come back?" It does not check the final answer.

Without a key, only keyword search is measured.

In [ ]:
import json

QUESTIONS_FILE = PROJECT_DIR / "eval_questions.json"
TOP_K = 3
MODES = ["keyword", "vector", "hybrid"]


def hit(expected_phrase, chunks):
    # A hit means the expected phrase appears in at least one of the top-k chunks (ignoring case).
    return any(expected_phrase.lower() in c.page_content.lower() for c in chunks)


questions = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))

modes = MODES
if not os.getenv("GEMINI_API_KEY", "").strip():
    modes = ["keyword"]
    print("GEMINI_API_KEY is not set. Only keyword search is measured.\n")

# Step 1 of the evaluation: for each mode, record whether the expected phrase is in the top-k chunks.
results = {}
for mode in modes:
    results[mode] = []
    for item in questions:
        chunks = retriever.search(item["question"], k=TOP_K, mode=mode)
        results[mode].append(hit(item["expected_phrase"], chunks))

# Step 2 of the evaluation: print a small table.
total = len(questions)
print(f"Retrieval hit rate (top {TOP_K}), {total} questions")
print(f"{'Mode':<10}{'Hits':>8}{'Rate':>8}")
for mode in modes:
    hits = sum(results[mode])
    print(f"{mode:<10}{hits:>5}/{total:<2}{hits / total:>8.0%}")

# Step 3 of the evaluation: list the questions that the last measured mode missed.
last_mode = modes[-1]
missed = [q["question"] for q, ok in zip(questions, results[last_mode]) if not ok]
print(f"\nMissed by {last_mode}:")
for question in missed or ["(none)"]:
    print(f" - {question}")

### Step 8: Check groundedness with a heuristic
**What:** `groundedness_check()` takes an answer and the retrieved text. It lists the key terms in the answer (long words, codes, and numbers) that are missing from the retrieved text. It flags the answer when less than 80 percent of the key terms are supported.
**Why:** A grounded answer uses words from its sources. An answer with invented details often has terms that are not in the sources.

**Limit:** This is a **simple heuristic, not a full evaluation**. It cannot tell whether a supported word is used correctly, and it misses paraphrases. Use it to spot obvious problems, not to certify answers.

In [ ]:
# Common words that carry little meaning. The groundedness check ignores them.
STOPWORDS = {
    "about", "after", "also", "because", "before", "could", "from", "have", "into",
    "only", "other", "should", "their", "there", "these", "they", "this", "those",
    "through", "what", "when", "where", "which", "while", "will", "with", "would",
    "your", "that", "than", "then", "were", "been", "being", "does", "more",
}

# Two hard-coded answers for the groundedness demo. They are NOT model output.
DEMO_QUESTION = "What does order code ACR-4471 include?"
DEMO_ANSWERS = {
    "supported by the FAQ": (
        "Order code ACR-4471 is the Starter Bundle. It includes the lamp, a spare shade, "
        "and a 1.5 m USB-C cable."
    ),
    "invented details": (
        "Order code ACR-4471 includes a lifetime extended warranty and free same-day "
        "delivery worldwide."
    ),
}


def key_terms(answer):
    # Key terms: longer words, plus codes and numbers. Stopwords and very short tokens are skipped.
    terms = set()
    for token in tokenize(answer):
        if token in STOPWORDS or len(token) < 2:
            continue
        if len(token) >= 5 or any(ch.isdigit() for ch in token):
            terms.add(token)
    return terms


def groundedness_check(answer, retrieved_chunks, min_support=0.8):
    # SIMPLE HEURISTIC, NOT A FULL EVALUATION.
    # It flags an answer when too many of its key terms are missing from the retrieved text.
    # It cannot tell whether a term is used with the right meaning, so read the flags with care.
    context_terms = set()
    for chunk in retrieved_chunks:
        context_terms.update(tokenize(chunk.page_content))
    terms = key_terms(answer)
    if not terms:
        return {"flagged": False, "support": None, "missing": []}
    missing = sorted(t for t in terms if t not in context_terms)
    support = 1 - len(missing) / len(terms)
    return {"flagged": support < min_support, "support": support, "missing": missing}


# Context comes from keyword search, so this step makes no API calls.
context = retriever.search(DEMO_QUESTION, k=TOP_K, mode="keyword")
print(f"\nGroundedness heuristic (demo, not a full evaluation). Question: {DEMO_QUESTION}")
for label, answer in DEMO_ANSWERS.items():
    check = groundedness_check(answer, context)
    status = "FLAGGED" if check["flagged"] else "ok"
    print(f" - {label}: {status}, support {check['support']:.0%}, missing terms: {check['missing']}")

### Step 5: Rewrite the question (optional)
**What:** `rewrite_query()` asks Gemini to turn the question into a search-friendly query.
**Why:** Users write chatty questions ("hey, how long is the guarantee on this?"). A short keyword query often retrieves better.

This step is optional and needs a key. It is not used in the evaluation by default. See Exercise 1 in the README.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

REWRITE_PROMPT = (
    "Rewrite the user's question so it works well as a search query over a product FAQ. "
    "Keep every code, number and product name exactly as written. Use plain keywords. "
    "Return only the rewritten query.\n\nQuestion: {question}"
)


def rewrite_query(question):
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, google_api_key=get_api_key(), temperature=0)
    reply = llm.invoke(REWRITE_PROMPT.format(question=question))
    return reply.content.strip()


print(rewrite_query("hey, how long is the guarantee on this lamp?"))

## Checkpoints and what you learned

- After Step 1: a chunk count (roughly 15 to 25, not checked by running), and text starting with `Q:`
- After Step 2: the keyword search returns the chunk that contains `Order code ACR-4471`
- After Step 3: vector search returns the chunk about the dead lamp and the outlet (needs a key)
- After Step 4: hybrid search returns a list that includes the battery pack entry (needs a key)
- After Step 7: a table with three rows (keyword, vector, hybrid), each with hits out of 10
- After Step 8: two groundedness lines, one for each demo answer

You learned that keyword search (BM25) finds exact words and codes, vector search finds meaning and paraphrases, reciprocal rank fusion combines two ranked lists, hit rate measures whether the right chunk was retrieved, and a groundedness heuristic is a quick check, not a full evaluation.

Run locally instead? See the README setup section.